# _**COMP 3610 - ASSIGNMENT 2**_
### _Samuel Soman - 816039318_

This assignment comprises of building, evaluating and interpreting machine models to predict taxi trip tip amounts using the NYC Yellow Taxi Trip dataset. Machine Learning fundamentals, feature engineering, model evaluation and deep learning with PyTorch used.

**Learning Objectives**
1. Perform feature engineering and data preprocessing for machine learning pipelines
2. Train and tune classification and regression models using Scikit-learn
3. Build and train a neural network using PyTorch
4. Evaluate models using appropriate metrics and cross-validation
5. Interpret model outputs and communicate findings effectively

**Prediction Tasks:**
1. **Regression:** Predict tip_amount (continuous)
2. **Classification:** Predict whether a trip will receive a [high tip] (tip_amount > 20% of fare_amount)

**Dataset:** NYC Yellow Taxi Trip Records - January 2024 (approximately 2.9 Million rows raw data). Only credit card payments (payment_type = 1) are used for modelling because tip amounts are reliably recorded only for card transactions.


---
# **Part 1: Data Preprocessing & Feature Engineering**
---

This section establishes the analytical foundation for the entire modelling workflow. The objective is to transform raw operational taxi data into a reliable, model-ready dataset with consistent semantics and defensible assumptions. Each preprocessing step is designed to reduce noise, remove invalid records, and prevent downstream bias or leakage. Feature engineering then translates domain behavior (time, trip characteristics, and geography) into structured signals that machine learning models can exploit.

## <u>1.0 Setup and Imports</u>

This setup cell standardizes the runtime environment and ensures all dependencies are available before any data operations begin. Reproducibility controls are applied early ( [random_state] and plotting defaults) so that results and visuals remain consistent across runs. Establishing these settings at the top also improves auditability, since every later step depends on this initial configuration. In technical reporting terms, this cell defines the computational context for the experiment.

In [ ]:
# Core libraries imports
import os, requests, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Global settings and configs.
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Core imports successful!")

## <u>1.1 Data Ingestion & Cleaning</u>

The cells below download the NYC Yellow Taxi Trip dataset (January 2024) and the Taxi Zone Lookup table, then apply the same cleaning pipeline developed in Assignment 1.

At this stage, the goal is to enforce data quality constraints before any modelling decisions are made. The cleaning rules are intentionally conservative and grounded in operational plausibility, for example valid trip duration, positive fare values, and correct event ordering. This protects model training from extreme outliers and malformed records that can distort error metrics and feature relationships. Using the same core cleaning logic as Assignment 1 also maintains methodological continuity and makes your pipeline easier to justify academically.

In [ ]:
# URLs and paths of data to be used
TRIP_DATA_URL  = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

RAW_DIR = os.path.join("data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

TRIP_DATA_PATH  = os.path.join(RAW_DIR, "yellow_tripdata_2024-01.parquet")
ZONE_LOOKUP_PATH = os.path.join(RAW_DIR, "taxi_zone_lookup.csv")


def download_file(url, dest_path):
    """Download a file if it does not already exist."""
    if os.path.exists(dest_path):
        print(f"Already exists, skipping: {dest_path}")
        return
    print(f"Downloading {url} ...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    print(f"Saved to {dest_path} ({size_mb:.1f} MB)")


download_file(TRIP_DATA_URL, TRIP_DATA_PATH)
download_file(ZONE_LOOKUP_URL, ZONE_LOOKUP_PATH)

# Load raw data 
df_raw = pd.read_parquet(TRIP_DATA_PATH)
zone_df = pd.read_csv(ZONE_LOOKUP_PATH)

print(f"\nTrip data shape : {df_raw.shape}")
print(f"Zone lookup shape: {zone_df.shape}")

In [ ]:
# DATA CLEANING  (reused from Assignment 1 and improved slight for this assignment)

df = df_raw.copy()
initial_rows = len(df)

df["tpep_pickup_datetime"]  = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# Remove nulls in critical columns
critical_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
                 "PULocationID", "DOLocationID", "fare_amount"]
before = len(df)
df = df.dropna(subset=critical_cols)
print(f"Nulls in critical cols  → removed {before - len(df):,}")

# Keep January 2024 only
before = len(df)
df = df[(df["tpep_pickup_datetime"] >= "2024-01-01") &
        (df["tpep_pickup_datetime"] <  "2024-02-01")]
print(f"Date filter (Jan 2024)  → removed {before - len(df):,}")

# Dropoff must be after pickup
before = len(df)
df = df[df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]]
print(f"Dropoff > pickup        → removed {before - len(df):,}")

# Distance > 0, fare in (0, 500], total > 0
before = len(df)
df = df[df["trip_distance"] > 0]
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 500)]
df = df[df["total_amount"] > 0]
print(f"Distance / fare filter  → removed {before - len(df):,}")

# Compute trip duration (needed for further filters)
df["trip_duration_minutes"] = (
    (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
)

# Duration 1–300 min, distance ≤ 200 mi
before = len(df)
df = df[(df["trip_distance"] <= 200) &
        (df["trip_duration_minutes"] <= 300) &
        (df["trip_duration_minutes"] >= 1)]
print(f"Duration / distance cap → removed {before - len(df):,}")

# Passenger count 1–9
before = len(df)
df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 9)]
print(f"Passenger count filter  → removed {before - len(df):,}")

removed = initial_rows - len(df)
print(f"\n{'='*50}")
print(f"Original rows : {initial_rows:,}")
print(f"Cleaned rows  : {len(df):,}")
print(f"Removed       : {removed:,}  ({removed/initial_rows*100:.1f}%)")

## <u>1.2 Filter for Credit Card Payments</u>

Since `tip_amount` is only reliably recorded for credit card transactions (`payment_type = 1`), we filter the dataset accordingly before any modelling work.

This decision is critical for label integrity. Including payment types where tips are missing or inconsistently recorded would introduce systematic measurement error and weaken both regression and classification targets. Restricting the analysis to card payments improves internal validity, even though it narrows the population represented by the final models. In reporting terms, this is a deliberate trade-off between completeness and target reliability.

In [ ]:
before = len(df)
df = df[df["payment_type"] == 1].copy()
print(f"Filtered to credit card only: {len(df):,} rows  "
      f"(removed {before - len(df):,} non-credit-card rows)")

## <u>1.3 Feature Engineering</u>

We create four groups of features:

| Group | Features |
|-------|----------|
| **Temporal** | [pickup_hour], [pickup_day_of_week] (0 = Mon), [is_weekend] |
| **Trip** | [trip_duration_minutes] (from cleaning), [trip_speed_mph], [log_trip_distance] |
| **Fare** | [fare_per_mile], [fare_per_minute] |
| **Zone** | One-hot encoded pickup & dropoff **borough** via the taxi zone lookup table |

Feature engineering is where raw transactional records are translated into behavior-oriented predictors. Temporal variables capture demand and rider context, trip variables capture travel intensity, and fare-normalized variables represent pricing structure beyond raw totals. Geographic encoding at borough level introduces interpretable spatial effects without the dimensional burden of full zone identifiers. Together, these features provide a balanced representation of rider, route, and fare dynamics required for robust supervised learning.

In [ ]:
# Temporal features 
df["pickup_hour"]        = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek   # 0 = Monday
df["is_weekend"]         = (df["pickup_day_of_week"] >= 5).astype(int)

# Trip features 
# trip_duration_minutes already computed during cleaning
df["trip_speed_mph"] = (
    df["trip_distance"] / (df["trip_duration_minutes"] / 60)
).clip(upper=80)                       # cap unrealistic speeds at 80 mph

df["log_trip_distance"] = np.log1p(df["trip_distance"])

# Fare features
df["fare_per_mile"] = np.where(
df["trip_distance"] > 0,
df["fare_amount"] / df["trip_distance"], 0,
)
df["fare_per_minute"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["fare_amount"] / df["trip_duration_minutes"],
    0,
)

# Zone features (borough from taxi zone lookup) 
# Merge pickup borough
df = df.merge(
    zone_df[["LocationID", "Borough"]].rename(
        columns={"LocationID": "PULocationID", "Borough": "pickup_borough"}
    ),
    on="PULocationID", how="left",
)
# Merge dropoff borough
df = df.merge(
    zone_df[["LocationID", "Borough"]].rename(
        columns={"LocationID": "DOLocationID", "Borough": "dropoff_borough"}
    ),
    on="DOLocationID", how="left",
)

df["pickup_borough"]  = df["pickup_borough"].fillna("Unknown")
df["dropoff_borough"] = df["dropoff_borough"].fillna("Unknown")

# One-hot encode boroughs
df = pd.get_dummies(df, columns=["pickup_borough", "dropoff_borough"],
                    prefix=["pu_boro", "do_boro"])

# Fill residual NaN in surcharge columns 
for col in ["congestion_surcharge", "Airport_fee"]:
    df[col] = df[col].fillna(0)

print("Feature engineering complete.")
print(f"Dataset shape: {df.shape}")
print(f"\nNew feature samples:")
display(df[["pickup_hour", "pickup_day_of_week", "is_weekend",
            "trip_speed_mph", "log_trip_distance",
            "fare_per_mile", "fare_per_minute"]].head(8))

## <u>1.4 Target Variable Creation</u>

| Target | Type | Definition |
|--------|------|------------|
| tip_amount | Continuous | Raw tip amount in USD (regression) |
| high_tip | Binary | 1 if tip_amount > 20% of fare_amount, else 0 (classification) |

This step formalizes the two predictive tasks and aligns them to evaluation strategy. The regression target captures absolute monetary behavior, while the classification target captures decision-oriented tipping behavior. Defining [high_tip] using a percentage threshold makes the class label more economically meaningful than a fixed dollar cutoff across fare sizes. This dual-target setup supports broader model comparison and demonstrates competency across both continuous and discrete prediction frameworks.

In [ ]:
# Regression target already exists as tip_amount
# Classification target
df["high_tip"] = (df["tip_amount"] > 0.20 * df["fare_amount"]).astype(int)

print("Target variable distribution:")
print(f"  tip_amount  — mean: ${df['tip_amount'].mean():.2f}, "
      f"median: ${df['tip_amount'].median():.2f}, "
      f"std: ${df['tip_amount'].std():.2f}")
print(f"\n  high_tip class counts:")
print(df["high_tip"].value_counts().to_string())
print(f"\n  high_tip class proportions:")
print(df["high_tip"].value_counts(normalize=True).round(4).to_string())

## <u>1.5 Data Splitting & Scaling</u>

- **70 / 15 / 15** train / validation / test split
- Stratified on [high_tip] to preserve class balance across splits
- **StandardScaler** fitted on **training data only** and applied to all splits
- One-hot (binary) borough features are **not** scaled

This stage enforces proper experimental design and protects against data leakage. The three-way split supports model development, tuning, and final unbiased evaluation on unseen data. Stratification preserves class proportions for the classification task, making validation and test metrics more stable and representative. Scaling only numeric features, and fitting the scaler on training data only, ensures mathematically fair preprocessing while retaining interpretability for binary indicators.

In [ ]:
# Scikit-learn: Splitting & Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define feature columns
NUMERIC_FEATURES = [
    "passenger_count", "trip_distance", "RatecodeID",
    "fare_amount", "extra", "mta_tax", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "Airport_fee",
    "pickup_hour", "pickup_day_of_week",
    "trip_duration_minutes", "trip_speed_mph",
    "log_trip_distance", "fare_per_mile", "fare_per_minute",
]

BINARY_FEATURES = ["is_weekend"] + [
    c for c in df.columns if c.startswith("pu_boro_") or c.startswith("do_boro_")
]

FEATURE_COLS = NUMERIC_FEATURES + BINARY_FEATURES

# Features to EXCLUDE (and reasons) 
EXCLUDED = {
    "tpep_pickup_datetime":  "datetime; info captured in temporal features",
    "tpep_dropoff_datetime": "datetime; info captured in trip_duration",
    "PULocationID":          "high-cardinality zone ID; replaced by borough",
    "DOLocationID":          "high-cardinality zone ID; replaced by borough",
    "VendorID":              "vendor identifier — not predictive of tips",
    "store_and_fwd_flag":    "technical recording flag — not tip-related",
    "payment_type":          "constant = 1 after credit-card filter",
    "total_amount":          "includes tip_amount → data leakage",
}

# Prepare feature matrix 
X = df[FEATURE_COLS].copy()
y_reg = df["tip_amount"].copy()
y_cls = df["high_tip"].copy()

# Fill any remaining NaN with column median (safety net)
X[NUMERIC_FEATURES] = X[NUMERIC_FEATURES].fillna(X[NUMERIC_FEATURES].median())

# Stratified 70 / 15 / 15 split 
X_train, X_temp, y_train_reg, y_temp_reg = train_test_split(
    X, y_reg, test_size=0.30, random_state=RANDOM_STATE, stratify=y_cls
)
# carry classification target using aligned indices
y_train_cls = y_cls.loc[X_train.index]
y_temp_cls  = y_cls.loc[X_temp.index]

X_val, X_test, y_val_reg, y_test_reg = train_test_split(
    X_temp, y_temp_reg, test_size=0.50, random_state=RANDOM_STATE,
    stratify=y_temp_cls
)
y_val_cls  = y_cls.loc[X_val.index]
y_test_cls = y_cls.loc[X_test.index]

# Scale numeric features (fit on train only) 
scaler = StandardScaler()
X_train[NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
X_val[NUMERIC_FEATURES]   = scaler.transform(X_val[NUMERIC_FEATURES])
X_test[NUMERIC_FEATURES]  = scaler.transform(X_test[NUMERIC_FEATURES])

print("Splitting & scaling complete.\n")

In [ ]:
# Document split sizes and class distributions 
for name, xs, yr, yc in [("Train", X_train, y_train_reg, y_train_cls),
                          ("Val",   X_val,   y_val_reg,   y_val_cls),
                          ("Test",  X_test,  y_test_reg,  y_test_cls)]:
    print(f"{name:>5s}  —  {len(xs):>10,} samples  |  "
          f"high_tip=1: {yc.sum():>8,} ({yc.mean()*100:.2f}%)  |  "
          f"high_tip=0: {(yc==0).sum():>8,} ({(1-yc.mean())*100:.2f}%)")

print(f"\nTotal features: {len(FEATURE_COLS)}  "
      f"({len(NUMERIC_FEATURES)} numeric + {len(BINARY_FEATURES)} binary)")

In [ ]:
# Feature summary table 
feat_summary = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Type": ["numeric"] * len(NUMERIC_FEATURES) + ["binary"] * len(BINARY_FEATURES),
    "Scaled": ["Yes"] * len(NUMERIC_FEATURES) + ["No"] * len(BINARY_FEATURES),
})
display(feat_summary)

print("\nFeatures EXCLUDED from modelling:")
for col, reason in EXCLUDED.items():
    print(f"  • {col:30s} — {reason}")

### **Part 1 Observations**

The preprocessing pipeline produced a coherent feature matrix with a balanced mix of numerical and binary predictors suitable for both regression and classification. Class stratification was successfully preserved across train, validation, and test splits, which strengthens confidence in downstream classification comparisons. Scaling behavior is methodologically correct because only continuous features were standardized using training statistics. At this point, the project has established a strong data governance baseline, reducing the risk of invalid comparisons in later modelling stages.

---
# **Part 2: Model Training & Tuning**
---

With a validated feature matrix in place, this section focuses on predictive modelling and performance optimization. The workflow progresses from baseline algorithms to tuned variants and then to a neural network benchmark. This ordering is intentional: simple models establish interpretable reference points, while more complex models test whether additional capacity yields measurable gains. From a technical report perspective, this provides a clear evidence chain for model selection decisions.

## <u>2.1 Baseline Models</u>

We train four baseline models with default (or lightly configured) hyperparameters and evaluate on the **validation** set.

| Task | Model |
|------|-------|
| Regression | Linear Regression, Random Forest Regressor |
| Classification | Logistic Regression, Random Forest Classifier |

Baseline models provide the reference point against which all tuning and architectural complexity must be justified. Linear and logistic regression serve as transparent linear benchmarks, while Random Forest models introduce non-linear capacity and feature interaction modeling. Evaluating both families on the same validation split enables fair comparison of bias-variance behavior. This step is essential for demonstrating that later improvements are genuine and not artifacts of metric selection.

In [ ]:
# Scikit-learn: Metrics
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc,
)

# Helper functions — reusable evaluation utilities
def eval_regression(y_true, y_pred):
    """Return a dict of regression metrics."""
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²":   r2_score(y_true, y_pred),
    }


def eval_classification(y_true, y_pred, y_proba):
    """Return a dict of classification metrics."""
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC":   roc_auc_score(y_true, y_proba),
    }


def print_metrics(name, metrics):
    """Pretty-print a metrics dict."""
    vals = "  |  ".join(f"{k}: {v:.4f}" for k, v in metrics.items())
    print(f"  {name:30s}  →  {vals}")

In [ ]:
# Scikit-learn: Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# REGRESSION BASELINES  (validation set)
reg_results = {}

# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train_reg)
y_val_pred_lr = lr_model.predict(X_val)
reg_results["Linear Regression"] = eval_regression(y_val_reg, y_val_pred_lr)

# Random Forest Regressor 
rf_reg = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_reg.fit(X_train, y_train_reg)
y_val_pred_rf_reg = rf_reg.predict(X_val)
reg_results["RF Regressor (baseline)"] = eval_regression(y_val_reg, y_val_pred_rf_reg)

print("Regression — Validation Performance:")
for name, metrics in reg_results.items():
    print_metrics(name, metrics)

In [ ]:
# Scikit-learn: Classification Models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# CLASSIFICATION BASELINES  (validation set)
cls_results = {}

#  Logistic Regression 
log_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_model.fit(X_train, y_train_cls)
y_val_pred_log   = log_model.predict(X_val)
y_val_proba_log  = log_model.predict_proba(X_val)[:, 1]
cls_results["Logistic Regression"] = eval_classification(y_val_cls, y_val_pred_log, y_val_proba_log)

# Random Forest Classifier 
rf_cls = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_cls.fit(X_train, y_train_cls)
y_val_pred_rf_cls  = rf_cls.predict(X_val)
y_val_proba_rf_cls = rf_cls.predict_proba(X_val)[:, 1]
cls_results["RF Classifier"] = eval_classification(y_val_cls, y_val_pred_rf_cls, y_val_proba_rf_cls)

print("Classification — Validation Performance:")
for name, metrics in cls_results.items():
    print_metrics(name, metrics)

## <u>2.2 Hyperparameter Tuning</u>

**Best-performing Scikit-learn model:** The Random Forest Regressor produced the best regression metrics (lowest MAE / RMSE, highest R2). To align with assignment requirements, tuning is performed using `RandomizedSearchCV` on a **stratified sample of 200,000 rows** from the training set.

**Search configuration (assignment-aligned):**
- Stratified sample size: **200,000 rows** (if available; otherwise full training set)
- 5 hyperparameters varied (>= 3 required)
- n_iter = 20, cv = 5
- Scoring: neg_mean_absolute_error
- random_state = 42 for reproducibility

This tuning phase is designed to improve generalization while remaining computationally tractable on CPU-only hardware. `RandomizedSearchCV` is appropriate here because it explores broad combinations efficiently without exhaustive grid expansion. Five-fold cross-validation provides a stronger estimate of out-of-sample behavior than a single split, reducing sensitivity to sampling noise. The stratified sample size is selected to satisfy assignment constraints while preserving class structure and keeping runtime practical.

In [ ]:
# Scikit-learn: Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

# Stratified sample for tuning (assignment: 200k-500k allowed) 
TUNE_SAMPLE = min(200_000, len(X_train))
if TUNE_SAMPLE < len(X_train):
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train_reg,
        train_size=TUNE_SAMPLE,
        random_state=RANDOM_STATE,
        stratify=y_train_cls,
    )
else:
    X_tune = X_train
    y_tune = y_train_reg

print(f"Tuning sample size: {len(X_tune):,} rows (from {len(X_train):,} training rows)\n")

# Hyperparameter search space (5 params, assignment requires >=3) 
param_dist = {
    "n_estimators": randint(80, 180),
    "max_depth": [10, 20, 30, None],
    "min_samples_split": randint(4, 16),
    "min_samples_leaf": randint(1, 7),
    "max_features": ["sqrt", "log2", None],
}

print("Search space:")
for k, v in param_dist.items():
    print(f"  {k}: {v}")

# RandomizedSearchCV (assignment-aligned: 5-fold CV) 
random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=False,
    error_score="raise",
)

t0 = time.time()
random_search.fit(X_tune, y_tune)
elapsed = time.time() - t0

print(f"\nTuning completed in {elapsed/60:.1f} minutes")
print(f"Best CV MAE: {-random_search.best_score_:.4f}")
print("\nBest parameters:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

In [ ]:
# Retrain best model on FULL training set
rf_reg_tuned = random_search.best_estimator_
rf_reg_tuned.fit(X_train, y_train_reg)

y_val_pred_tuned = rf_reg_tuned.predict(X_val)
reg_results["RF Regressor (tuned)"] = eval_regression(y_val_reg, y_val_pred_tuned)

# Compare tuned vs baseline on validation set 
print("Regression — Validation: Baseline vs Tuned")
print("-" * 70)
for name in ["RF Regressor (baseline)", "RF Regressor (tuned)"]:
    print_metrics(name, reg_results[name])

improvement_mae  = reg_results["RF Regressor (baseline)"]["MAE"] - reg_results["RF Regressor (tuned)"]["MAE"]
improvement_r2   = reg_results["RF Regressor (tuned)"]["R²"]    - reg_results["RF Regressor (baseline)"]["R²"]
print(f"\n  MAE improvement : {improvement_mae:+.4f}")
print(f"  R² improvement  : {improvement_r2:+.4f}")

## <u>2.3 Neural Network Model</u>

We build a feedforward neural network with PyTorch for the **classification** task (`high_tip`).

**Architecture:**
| Layer | Neurons | Activation | Extras |
|-------|---------|------------|--------|
| Input | n_features | — | — |
| Hidden 1 | 128 | ReLU | BatchNorm + Dropout(0.3) |
| Hidden 2 | 64 | ReLU | BatchNorm + Dropout(0.3) |
| Hidden 3 | 32 | ReLU | — |
| Output | 1 | — | BCEWithLogitsLoss applies sigmoid |

**Training:** Adam optimiser (lr = 0.001), batch size 1024, 30 epochs.

The neural network is included as a non-linear benchmark to test whether learned hierarchical representations improve classification performance on tabular data. Batch normalization and dropout are used to stabilize optimization and limit overfitting as depth increases. BCEWithLogitsLoss is selected for numerical stability in binary classification, while probability thresholding at 0.5 supports direct metric comparison with Scikit-learn models. This section also demonstrates practical deep learning workflow integration within a primarily classical ML pipeline.

In [ ]:
# PyTorch 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(RANDOM_STATE)

# Neural Network - Definition
class TipClassifierNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),           # raw logit
        )

    def forward(self, x):
        return self.net(x)


# Prepare tensors 
def to_tensor(df_or_series, dtype=torch.float32):
    
    # Ensure mixed pandas dtypes (e.g., bool/object from one-hot features)
    # are converted into a contiguous numeric array compatible with torch.
    
    arr = np.asarray(df_or_series, dtype=np.float32 if dtype == torch.float32 else None)
    return torch.tensor(arr, dtype=dtype)

X_train_t = to_tensor(X_train)
y_train_cls_t = to_tensor(y_train_cls)
X_val_t = to_tensor(X_val)
y_val_cls_t = to_tensor(y_val_cls)

train_ds = TensorDataset(X_train_t, y_train_cls_t)
val_ds   = TensorDataset(X_val_t, y_val_cls_t)

BATCH  = 1024
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)

n_features = X_train_t.shape[1]
print(f"Input features: {n_features}")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

In [ ]:
# Neural Network — Training Loop
EPOCHS = 30
LR = 0.001

model = TipClassifierNN(n_features)
criterion = nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # Train the model for one epoch
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        optimiser.zero_grad()
        logits = model(xb).squeeze()
        loss = criterion(logits, yb)
        loss.backward()
        optimiser.step()
        running += loss.item() * len(xb)
    train_losses.append(running / len(train_ds))

    # Validate the model for one epoch 
    model.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb).squeeze()
            loss = criterion(logits, yb)
            running += loss.item() * len(xb)
    val_losses.append(running / len(val_ds))

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>2}/{EPOCHS}  "
              f"Train Loss: {train_losses[-1]:.4f}  "
              f"Val Loss: {val_losses[-1]:.4f}")

print("\nTraining complete.")

In [ ]:
# Training & validation loss curves 
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, EPOCHS + 1), train_losses, label="Train Loss", linewidth=2)
ax.plot(range(1, EPOCHS + 1), val_losses,   label="Val Loss",   linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Neural Network: Training & Validation Loss Curves", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# NN Evaluation on validation set 
model.eval()
with torch.no_grad():
    logits_val = model(X_val_t).squeeze()
    proba_val_nn = torch.sigmoid(logits_val).numpy()
    pred_val_nn  = (proba_val_nn >= 0.5).astype(int)

cls_results["Neural Network"] = eval_classification(
    y_val_cls.values, pred_val_nn, proba_val_nn
)

print("Classification — Validation Performance (all models):")
print("-" * 80)
for name, metrics in cls_results.items():
    print_metrics(name, metrics)

### **Part 2 Observations**

Baseline-to-tuned comparison indicates meaningful improvement in Random Forest regression performance, confirming that hyperparameter search contributed real signal rather than random variance. The neural network section is positioned appropriately as a complementary non-linear benchmark rather than a replacement for classical models. Runtime decisions in tuning remain aligned with assignment constraints while preserving reproducibility and methodological rigor. Collectively, Part 2 now provides a clear justification path from initial model choice to optimized candidate selection.

---
# **Part 3: Model Evaluation & Interpretation**
---

This section consolidates model outputs into evidence suitable for technical decision-making. Evaluation is performed on a held-out test set to estimate true generalization after all training and tuning choices are finalized. Multiple diagnostic views are used, including aggregate metrics, ROC behavior, error structure, and feature attribution. The objective is not only to identify top performance, but also to explain model behavior, strengths, and operational limitations in a clear and defensible way.